# rlmflow visualization walkthrough

A guided tour of every visualization that ships with rlmflow. Everything here renders **inline in Jupyter** and runs **offline** against the saved boids workspace under `examples/data/boids-sim-workspace/` — no API keys, no LLM calls.

Every viewer / viz function takes a unified **`source`**: a `Workspace`, a workspace path, a single `Graph`, an iterable of `Graph`s, or a path to a graph dump. So you can hand any of these the same workspace path string and skip the loading boilerplate. For the underlying `Graph` query API (walk, find, where, session), see [`node_basics.ipynb`](./node_basics.ipynb).

**What we cover**

1. Loading a workspace
2. Inline tree rendering — `graph.tree()`, `ascii_boxes`
3. **Interactive viewer** — `open_viewer` (Gradio: clickable graph, slider, per-agent chat)
4. Plotly graph + HTML stepper — `graph.plot()`, `save_html`, `save_steps`
5. Topology renders — Mermaid (state / flowchart / sequence), DOT, D2
6. Step-indexed timeline — `gantt`, `gantt_html`
7. Per-state detail — `code_log`, `error_summary`, `message_stream`, `diff_system_prompts`
8. Cost & reports — `token_sparkline`, `budget_burndown`, `report_md`
9. Comparison across runs — `bench_table`
10. CLI equivalents

## 1. Load a workspace

`Workspace.open_path(...)` opens a saved run. `load_steps()` retraces the run as one `Graph` per state-append; `load_graph()` returns just the final snapshot. Most viz functions also take the workspace path directly — we'll mix both styles below to show the unified `source` API.

In [ ]:
from rlmflow.workspace import Workspace

WS_PATH = "../data/boids-sim-workspace"
ws = Workspace.open_path(WS_PATH)
graphs = ws.load_steps()
final = graphs[-1]
print(f"loaded {len(graphs)} snapshots  \u00b7  final has {len(final.agents)} agents")

## 2. Inline tree rendering

`graph.tree()` returns a per-agent timeline as plain text. `ascii_boxes` is a richer variant that wraps each agent in a Rich panel — better for pasting into PR comments or terminal screenshots.

In [ ]:
print(final.tree())

In [ ]:
from rlmflow.utils.viz import ascii_boxes

# `source` accepts a Graph, Workspace, path, or list[Graph] — try the path:
print(ascii_boxes(WS_PATH))

## 3. Interactive viewer

The full interactive graph + per-agent chat lives in `open_viewer`. It embeds inline in Jupyter via Gradio, with a step slider, clickable nodes, and a color-coded conversation panel for any agent you click.

For static export (PR comments, email), use the Plotly stepper in section 4 or the topology renders in section 5.

In [ ]:
from rlmflow.utils.viewer import open_viewer

# `open_viewer` takes the same `source` — workspace, path, graph, or list of graphs.
open_viewer(
    WS_PATH,
    inline=True,
    height=720,
    quiet=True,
)

## 4. Plotly graph + HTML stepper

`graph.plot()` builds a Plotly figure with every node laid out as a tidy tree (one column per branching child, edges from `Graph.edges`). `save_html(source, out)` stitches every snapshot into a self-contained HTML page with arrow-key navigation; `save_steps(source, out_dir)` writes one PNG per step (handy for slide decks). All take the unified `source`.

In [ ]:
final.plot(height=500)

In [ ]:
import tempfile
from pathlib import Path
from IPython.display import IFrame
from rlmflow.utils.viewer import save_html

out = Path(tempfile.gettempdir()) / "rlmflow-stepper.html"
save_html(WS_PATH, out)
print(f"wrote {out}")
IFrame(str(out), width="100%", height=720)

## 5. Topology renders

Static topology renders for embedding in docs, PRs, post-mortems. Mermaid blocks render inline via `IPython.display.Markdown`. The same renders are reachable from the CLI as `rlmflow render <path> -f F`.

In [ ]:
from IPython.display import Markdown
from rlmflow.utils.export import to_mermaid

Markdown(f"```mermaid\n{to_mermaid(final)}\n```")

In [ ]:
from rlmflow.utils.export import to_mermaid_flowchart

Markdown(f"```mermaid\n{to_mermaid_flowchart(final)}\n```")

In [ ]:
from rlmflow.utils.export import to_mermaid_sequence

Markdown(f"```mermaid\n{to_mermaid_sequence(final)}\n```")

In [ ]:
from rlmflow.utils.export import to_dot, to_d2

print("--- DOT (paste into a Graphviz renderer) ---")
print(to_dot(final))
print()
print("--- D2 (https://d2lang.com) ---")
print(to_d2(final))

## 6. Step-indexed timeline

`gantt(graphs)` prints a Rich table — one row per agent, one column per step, single-letter glyphs (`Q`/`A`/`O`/`S`/`R`/`F`/`E`). `gantt_html(graphs)` writes a colorful self-contained HTML page; we render it inline below.

In [ ]:
from rlmflow.utils.viz import gantt

gantt(WS_PATH)

In [ ]:
from IPython.display import HTML
from rlmflow.utils.viz import gantt_html

HTML(gantt_html(WS_PATH, title="boids run"))

## 7. Per-state detail

Drill into the actual code that ran, errors that happened, and the chat-log view for any agent.

In [ ]:
from rlmflow.utils.viz import code_log

print(code_log(final))

In [ ]:
# The boids run actually has one error (a child that crashed and retried).
# `error_summary` groups errors by kind across the whole subtree.
from rlmflow.utils.viz import error_summary

print(error_summary(WS_PATH))

In [ ]:
from rlmflow.utils.viz import message_stream

stream = message_stream("root.boid_js", final)
print(stream[:1800] + ("\n..." if len(stream) > 1800 else ""))

In [ ]:
from rlmflow.utils.viz import diff_system_prompts

children = sorted(aid for aid in final if aid != "root")
print("child agents:", children)
a, b = children[0], children[1]
print(f"\ndiffing: {a}  vs  {b}\n")
diff = diff_system_prompts(final, final, a)  # same graph, so prompts match
print(diff)

## 8. Cost & reports

Three views: a one-line ASCII sparkline of cumulative tokens, a budget burndown bar, and a full Markdown report bundling tree + cost + result + errors.

In [ ]:
from rlmflow.utils.viz import budget_burndown, token_sparkline

print(token_sparkline(WS_PATH))
print(budget_burndown(WS_PATH))
print(budget_burndown(WS_PATH, max_budget=100_000))

In [ ]:
from rlmflow.utils.viz import report_md

Markdown(report_md(WS_PATH, title="boids \u2014 coding-agent run"))

## 9. Comparison across runs

`bench_table` aggregates labeled traces — use it to compare benchmark runs side by side (cost, duration, outcome, errors).

In [ ]:
from rlmflow.utils.viz import bench_table

# `bench_table` accepts a mapping of label -> source. Each source can be a
# workspace path, a Workspace, a Graph, or a list of Graphs. We fake two runs
# by trimming the same workspace's snapshots to show the table shape.
print(
    bench_table(
        {
            "boids:full": WS_PATH,
            "boids:firsthalf": graphs[: len(graphs) // 2 + 1],
        }
    )
)

## 10. CLI equivalents

Every render here is also available without writing Python. From a shell:

```bash
rlmflow view   examples/data/boids-sim-workspace
rlmflow render examples/data/boids-sim-workspace -f mermaid-flowchart
rlmflow render examples/data/boids-sim-workspace -f gantt-html  -o gantt.html
rlmflow render examples/data/boids-sim-workspace -f report-md   -o report.md
rlmflow render examples/data/boids-sim-workspace -f code-log
rlmflow render examples/data/boids-sim-workspace -f tokens
```

All formats: `mermaid` \u00b7 `mermaid-flowchart` \u00b7 `mermaid-sequence` \u00b7 `dot` \u00b7 `d2` \u00b7 `tree` \u00b7 `ascii-boxes` \u00b7 `gantt-html` \u00b7 `report-md` \u00b7 `code-log` \u00b7 `error-summary` \u00b7 `tokens`.